# **D502 Data Analytics Capstone**

## Ditte Joergensen

Research Question: Is there a relationship between flight distance and arrival delay in U.S. domestic flights?

### Data Wrangling

This section will be loading in the dataset and exploring the structure of it. This will include looking for null or missing values, duplicates and wrong datatypes. The cleaning of the dataset will also be included in this section.

In [ ]:
# import necessary packages and libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr

In [ ]:
# download flight dataset into notebook
flights_raw = pd.read_csv("flights_sample_3m.csv")

# make copy of dataset for cleaning
flights = flights_raw.copy()

In [ ]:
flights.head()

In [ ]:
flights.info()

In [ ]:
flights.shape

In [ ]:
flights.isna().sum()

In [ ]:
flights.describe()

In [ ]:
flights.duplicated().sum()

In [ ]:
flights.sample(10)

When looking through the dataset, there are some areas that need special attention.
- FL_DATE is currently set as string type
- The dataset contains cancelled flights, these will not be needed for my analysis
- Diverted flights will not be included either
- Columns containing information about the flight itself, city, time-of-day and elapsed time will not be needed
- A category column for distance would be useful for discriptive and exploratory analysis

In [ ]:
# Convert FL_DATE to date column
flights['FL_DATE'] = pd.to_datetime(flights['FL_DATE'])

flights['FL_DATE'].info()

In [ ]:
# Remove cancelled flights from dataset
flights = flights[flights['CANCELLED'] == 0]

In [ ]:
# Remove diverted flights from dataset
flights = flights[flights['DIVERTED'] == 0]

In [ ]:
# Drop NaN values from DISTANCE and ARR_DELAY columns
flights = flights.dropna(subset=['DISTANCE', 'ARR_DELAY'])

In [ ]:
flights.isna().sum()

In [ ]:
# Funtion to create distance categories
def add_distance_cats(df):
    bins = [0, 500, 1500, df['DISTANCE'].max()]
    labels = ['Short (0-500 miles)', 'Medium (501-1500)', 'Long (1501+ miles)']
    df['DISTANCE_GROUP'] = pd.cut(df['DISTANCE'], bins=bins, labels=labels)
    return df

In [ ]:
# Create distance categories
flights = add_distance_cats(flights)

In [ ]:
# Keep only necessary columns from dataset
columns_to_keep = ['FL_DATE', 'AIRLINE', 'ORIGIN', 'DEST', 'DISTANCE', 'ARR_DELAY', 'DELAY_DUE_CARRIER', 
'DELAY_DUE_WEATHER', 'DELAY_DUE_NAS', 'DELAY_DUE_SECURITY', 'DELAY_DUE_LATE_AIRCRAFT', 'DISTANCE_GROUP']

flights = flights[columns_to_keep].copy()

In [ ]:
flights.sample(5)

### Descriptive Analysis

This section will take a closer look at the dataset. Specifically, it will be looking at summary statistics such as mean, standard deviation, minimum and maximum. The main columns that will be looked at is DISTANCE, ARR_DELAY and DISTANCE_GROUP as these are most important to this project. Visualizations will be included to understand the data better. 

In [ ]:
# Look at summary statistics for arrival delay
flights['ARR_DELAY'].describe()

In [ ]:
# look at summary statistics for distance
flights['DISTANCE'].describe()

In [ ]:
# Function to find mean of a column
def find_mean(df, column):
    mean_value = df[column].mean()
    print(f"The mean of {column} is: {mean_value:.2f} minutes")
    return mean_value

In [ ]:
# Find mean value for all delay columns
mean_carrier_delay = find_mean(flights, "DELAY_DUE_CARRIER")
mean_weather_delay = find_mean(flights, "DELAY_DUE_WEATHER")
mean_NAS_delay = find_mean(flights, "DELAY_DUE_NAS")
mean_security_delay = find_mean(flights, "DELAY_DUE_SECURITY")
mean_late_aircraft_delay = find_mean(flights, "DELAY_DUE_LATE_AIRCRAFT")

In [ ]:
# reusable function to add title and labels to a visual
def add_labels(title, xlabel, ylabel):
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.show()

In [ ]:
# Distribution of ARR_DELAY visualized with a histogram
flights['ARR_DELAY'].plot(kind='hist', bins=50, figsize=(8,5))
add_labels('Distribution of Arrival Delays', 'Arrival Delay (Minutes)', 'Number of Flights')

This histogram shows that arrival delays are extremely right-skewed, with a majority of the flights clustered near zero minutes of delay. Most flights arrive on time or only slightly late or early, this created the tall spike at the left side of the distribution. Only a very few number of flights experience long or extreme delays. The long tail stretching to the right shows rare, but very large delay events which inflate the overall spread of the data. 

In [ ]:
# Remove top 1% in arrival delay histogram
arrival_delays_trimmed = flights[flights['ARR_DELAY'] < flights['ARR_DELAY'].quantile(0.99)]
arrival_delays_trimmed['ARR_DELAY'].plot(kind='hist', bins=50, figsize=(8,5))
add_labels('Distribution of Arrival Delays (Top 1% Removed)', 'Arrival Delay (Minutes)', 'Number of Flights')

Removing the top 1% of extreme delays reveals the true and everyday shape of arrival-delay behavior. The histogram becomes much more interpretable, most flights cluster tightly around zero with a clear peak just below on-time arrival. The right-skew shows that postitive delays happen slightly more than early arrivals. This trimmed view highlights that the typical delays fall within a narrow range. 

In [ ]:
# Bar chart of distance group counts
flights['DISTANCE_GROUP'].value_counts().sort_index().plot(kind='bar')
add_labels('Number of Flights by Distance Group', 'Distance Group', 'Count of Flights')

This bar chart shows the amount of flights that fall into each distance category, short/medium/long. It is clear from the graph that medium distance flights (501-1500) are the most common, around 1.5 million flights at this distance. Short distance flights (0-500) are the second most common with just over 1 million flights. Long distance flights (1501+) are the least common with roughly 350,000 flights. This graph indicates that regional and mid-range routes domincate the air travel. 

### Exploratory Analysis

This section will focus on exploring the relationship between distance and arrival delays. A scatterplot and boxplot visualization will be included. 

In [ ]:
# Scatterplot of relationship between distance and arrival delay
plt.figure(figsize=(8,5))
sns.scatterplot(data=flights, x='DISTANCE', y='ARR_DELAY', alpha=0.3)
add_labels('Scatterplot of Distance vs Arrival Delay', 'Distance (Miles)', 'Arrival Delay (Minutes)')

This scatterplot shows the relationship bwteen flight distance (x-axis) and arrival delay in minutes (y-axis). It appears that there s no clear upwards trend as distance increases, this means that longer flights to not consistently have larger arrival delays. Most delays are small, the majority of points cluster close to 0 minutes in delay. It can also be seen that extreme delays happen mostly on short/medium flights, very large delays such as over 1000 minutes appear mostly under about 3000 miles. 

In [ ]:
# Boxplot of arrival delay by distance group
plt.figure(figsize=(8,5))
sns.boxplot(data=flights, x='DISTANCE_GROUP', y='ARR_DELAY')
plt.ylim(-50, 300)
add_labels('Arrival Delay by Distance Group', 'Distance Group', 'Arrival Delay (Minutes)')

In the boxplot above, the y-axis was limited to -50 to 300 to exclude extreme outliers as they lessened the readability of the visualization. This adjustment preserves all data while allowing the distribution of arrival delays to be clearly visualized. It looks like both the median and interquartile ranges are very similar for all three categories. They all have medians slightly below 0 which indicates that the typical flight arrives a little early, regardless of distance. The middle 50% of delays is similar across all distances. Long-distance flights appear to have a slightly wider range within this visible visual. 

In [ ]:
# Compare mean and median arrival delays by distance group
flights.groupby('DISTANCE_GROUP')['ARR_DELAY'].agg(['mean', 'median', 'std', 'count'])

### Correlation Test

In [ ]:
r, p = pearsonr(flights['DISTANCE'], flights['ARR_DELAY'])
print("Correlation coeffecient (r):", r)
print("P-value:", p)

According to Pearson's correlation test. The coeffecient is extremely close to zero (~0.00188), which means that there is no meaningful linear relationship between flight distance and arrival delay. As distance increases, delay minutes do not increase or decrease. The p-value is below 0.05, so the test is statistically significant. This significance, however, is most likely driven by the very large sample size with over 2 million flights. 

In [ ]:
# Convert notebook to html
!python -m nbconvert --to html analysis.ipynb